# Example 5: Reverberation Enhancement System (RES)

This notebook shows how to build a complete **Reverberation Enhancement System** in PyRES.

An RES consists of:
- A **physical room** (the actual acoustic space, here modelled with decaying white noise)
- A **virtual room** (the DSP that processes microphone signals to drive the loudspeakers)
- A **system gain** \( G \) that controls the overall loop gain

The `RES` class combines these components and provides methods to:
- Simulate the system (natural vs. enhanced response)
- Compute the gain before instability (GBI)
- Analyse the open‑loop and closed‑loop transfer matrices and their eigenvalues

We reuse the physical room from Example 2 (`PhRoom_wgn`) and the virtual room from Example 1 (`FDN`).

## 1. Imports and Path Setup

In [ ]:
import sys
import os
# Add parent directory to path (so that PyRES can be imported)
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import matplotlib.pyplot as plt
from flamo.functional import mag2db
from PyRES.virtual_room import FDN
from PyRES.physical_room import PhRoom_wgn
from PyRES.res import RES
from PyRES.plots import plot_spectrograms_compare, plot_evs_distribution

## 2. Time–Frequency Parameters

These are the same as before: sampling rate, FFT size, and anti‑aliasing decay.

In [ ]:
samplerate = 48000          # Hz
nfft = samplerate * 3       # FFT size (3 seconds)
alias_decay_db = 0          # No extra anti‑aliasing decay

## 3. Create the Physical Room (stochastic)

We use `PhRoom_wgn` with the same dimensions and reverberation time as in Example 2.

In [ ]:
room_dims = (12.1, 8.5, 3.2)   # length, width, height (m)
room_RT = 0.7                   # reverberation time (s)
n_M = 4                         # number of system microphones
n_L = 8                         # number of system loudspeakers

physical_room = PhRoom_wgn(
    fs=samplerate,
    nfft=nfft,
    alias_decay_db=alias_decay_db,
    room_dims=room_dims,
    room_RT=room_RT,
    n_M=n_M,
    n_L=n_L
)

## 4. Create the Virtual Room (FDN)

We use `FDN` with the same number of inputs/outputs as the physical room, and frequency‑dependent reverberation times.

In [ ]:
virtual_room = FDN(
    n_M=n_M,
    n_L=n_L,
    fs=samplerate,
    nfft=nfft,
    alias_decay_db=alias_decay_db,
    order=16,
    t60_DC=1.0,
    t60_NY=0.2,
)

## 5. Build the Reverberation Enhancement System

The `RES` class simply takes the two room objects as arguments. It automatically computes an initial safe system gain \( G \) (set to GBI – 3 dB by default) to ensure stability.

In [ ]:
res = RES(
    physical_room=physical_room,
    virtual_room=virtual_room
)

print(f"\nThe RES class is a container for the PhRoom and VrRoom class instances:")
print(f"  Physical room: {type(res.phroom).__name__}")
print(f"  Virtual room: {type(res.vrroom).__name__}")
print(f"The RES class also hosts the system gain G, which is the master gain of the audio setup.")
print(f"At first the system gain is set to GBI - 3 dB, where GBI is an estimation of the gain before instability.")
print(f"  System gain G: {type(res.G)}, with value {mag2db(res.G.param[0]):.2f} dB")

## 6. Simulate the System (RES off vs. RES on)

The method `system_simulation()` returns three sets of impulse responses:
- `nat_rirs`: natural path (stage → audience) – system off
- `ea_rirs`: electroacoustic path only (stage → microphones → virtual room → loudspeakers → audience) – neglecting the natural path
- `res_rirs`: complete system (natural + electroacoustic) – system on

We compare the natural and enhanced responses using spectrograms.

In [ ]:
nat_rirs, _, res_rirs = res.system_simulation()

plot_spectrograms_compare(
    ir_1=nat_rirs[:2*samplerate, 0],
    ir_2=res_rirs[:2*samplerate, 0],
    fs=samplerate,
    nfft=2**11,
    noverlap=2**10,
    label1='RES off',
    label2='RES on'
)
plt.show()

## 7. Gain Before Instability (GBI)

The GBI is estimated from the open‑loop transfer matrix. By default a conservative criterion (`'eigenvalue_magnitude'`) is used, but you can also use `'eigenvalue_real_part'` for a less conservative estimate.

In [ ]:
print(f"\nEstimated GBI (eigenvalue magnitude criterion): {mag2db(res.compute_GBI(criterion='eigenvalue_magnitude')):.2f} dB")

# We can set the gain exactly to the estimated GBI (though it may still be stable because the estimate is conservative).
res.set_G_to_GBI(gbi_estimation_criterion='eigenvalue_magnitude')
print(f"System gain G set to GBI: {mag2db(res.G.param[0]):.2f} dB")

# Re‑run the simulation with the higher gain
nat_rirs, _, res_rirs = res.system_simulation()

plot_spectrograms_compare(
    ir_1=nat_rirs[:2*samplerate, 0],
    ir_2=res_rirs[:2*samplerate, 0],
    fs=samplerate,
    nfft=2**11,
    noverlap=2**10,
    label1='RES off',
    label2='RES on (G = GBI)'
)
plt.show()

## 8. Open‑Loop and Closed‑Loop Analysis

The `RES` class provides several methods to inspect the system's internal structure:
- `open_loop()` returns the processing chain of the open‑loop transfer matrix (virtual room → gain → physical room feedback path).
- `open_loop_responses()` returns the impulse and frequency responses of the open‑loop matrix.
- `closed_loop()` returns a recursive processing block representing the closed‑loop system.
- `closed_loop_responses()` returns the impulse and frequency responses of the closed‑loop matrix.
- `open_loop_eigenvalues()` computes the eigenvalues of the open‑loop matrix as a function of frequency.

In [ ]:
print(f"\nOpen‑loop processing chain:")
for module in res.open_loop()._modules.values():
    print(f"  {type(module).__name__}")

# Get impulse and frequency responses of the open‑loop matrix
ol_ir, ol_fr = res.open_loop_responses()
print(f"\nOpen‑loop impulse responses shape: {ol_ir.shape}")
print(f"Open‑loop frequency responses shape: {ol_fr.shape}")

print(f"\nClosed‑loop is a recursive block: {type(res.closed_loop()).__name__}")
print("Its feedforward path contains:")
for module in res.closed_loop().feedforward._modules.values():
    print(f"  {type(module).__name__}")
print(f"And its feedback path is: {type(res.closed_loop().feedback).__name__}")

cl_ir, cl_fr = res.closed_loop_responses()
print(f"\nClosed‑loop impulse responses shape: {cl_ir.shape}")
print(f"Closed‑loop frequency responses shape: {cl_fr.shape}")

evs = res.open_loop_eigenvalues()
print(f"\nOpen‑loop eigenvalues shape (frequency bins × channels): {evs.shape}")

## 9. Visualise the Open‑Loop Eigenvalue Distribution

The eigenvalues of the open‑loop matrix provide insight into the energy flow and potential instability. A flat magnitude distribution across frequency indicates even energy distribution.

In [ ]:
plot_evs_distribution(
    evs=evs,
    fs=samplerate,
    nfft=nfft,
    lower_f_lim=20,
    higher_f_lim=20000
)
plt.show()

## 10. Conclusion

You have successfully built a complete Reverberation Enhancement System in PyRES, combining a synthetic physical room with an FDN‑based virtual room. The notebook demonstrated:
- System simulation with different gains
- Estimation of the gain before instability
- Examination of open‑loop and closed‑loop properties
- Visualisation of eigenvalue distributions

To explore further, try different physical room models (e.g., from the DataRES dataset), change the virtual room parameters, or use a different GBI estimation criterion.